# Cleaning `NBA_player_stats.csv`

This notebook cleans the full ESPN feature table that `scripts/build_dataset.py` writes to `data/NBA_player_stats.csv` and saves the result to `data/NBA_player_stats_cleaned.csv`. It never modifies the original file.

The cleaning follows the EDA findings. In short:

| step | what | why |
|---|---|---|
| 1 | Drop rows with 0 minutes played | Did-not-play rows. They have no usage %, some carry stray stats, and a few are marked as starters |
| 2 | Give each player one `espn_player_id` | One player had two ESPN IDs |
| 3 | Collapse `position` to G / F / C | The detailed codes (PG, SG, SF, PF) fade out after 2021 |
| 4 | Clip rate features that blow up on tiny samples | Usage % reached 196 and points per minute reached 3.0 after 1-minute stints |
| 5 | Cap `days_since_last_game` at 30 | Offseason gaps ran to 1,559 days |
| 6 | Winsorize team and opponent ratings | Early-season ratings come from 1-3 games |

**What stays the same on purpose**

- The columns, their order and their dtypes are unchanged. `full_features.feature_columns()` treats every non-identity, non-`target_` column as a model feature, so a new column here would silently become a feature in the training notebooks.
- Missing values are left as NaN. They almost always mean "no history yet" (see the last section), and XGBoost handles NaN directly.
- Same-game minutes are never used to filter beyond the 0-minute rows, because filtering on a same-game value would bias which rows the model trains on.

## Config

- `INPUT_PATH` / `OUTPUT_PATH`: the source CSV and the cleaned copy.
- `SAVE_OUTPUT`: set to `False` to run the notebook without writing a file.
- `MIN_TARGET_MINUTES`: rows with `target_min` at or below this are dropped. `0` removes only the 0-minute rows.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Locate the repo root from the notebook's folder or any parent, and make it importable.
REPO_ROOT = next(
    (root for root in (Path.cwd(), *Path.cwd().parents) if (root / 'data' / 'processing' / 'full_features.py').exists()),
    None,
)
assert REPO_ROOT is not None, 'Could not find the repo root (data/processing/full_features.py)'
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from data.processing import full_features

INPUT_PATH = REPO_ROOT / 'data' / 'NBA_player_stats.csv'
OUTPUT_PATH = REPO_ROOT / 'data' / 'NBA_player_stats_cleaned.csv'
SAVE_OUTPUT = True
MIN_TARGET_MINUTES = 0

assert INPUT_PATH.exists(), f'{INPUT_PATH} not found. Run scripts/build_dataset.py first.'
print(INPUT_PATH)
print(OUTPUT_PATH)

/Users/swayamshree/Desktop/Capstone Project/CPSC491-02Group8SportsBetting/data/NBA_player_stats.csv
/Users/swayamshree/Desktop/Capstone Project/CPSC491-02Group8SportsBetting/data/NBA_player_stats_cleaned.csv


## Load the data

`game_id` is read as a string. `game_date` stays a string in `YYYY-MM-DD` form, so the saved file keeps the original format; it is converted to a date only for checks. The original frame is kept as `raw` so every step can be compared with it.

In [2]:
raw = pd.read_csv(INPUT_PATH, dtype={'game_id': str})
df = raw.copy()

ORIGINAL_COLUMNS = list(raw.columns)
ORIGINAL_DTYPES = raw.dtypes.copy()
ORIGINAL_FEATURES = full_features.feature_columns(raw.columns)
ORIGINAL_SPLIT_BY_SEASON = raw.groupby('season')['split'].first().to_dict()

# The file is written in date order; cleaning keeps that order (steps only drop rows or edit values).
assert pd.to_datetime(raw['game_date']).is_monotonic_increasing, 'Input is not in date order'

# One row per cleaning step, shown in the summary at the end.
log = []


def record(step, rows_dropped=0, values_changed=0, note=''):
    log.append({'step': step, 'rows dropped': rows_dropped, 'values changed': values_changed, 'note': note})


print(f'{raw.shape[0]:,} rows x {raw.shape[1]} columns')
print(f"{raw['game_id'].nunique():,} games, {raw['player_id'].nunique():,} players, seasons {sorted(raw['season'].unique())}")
raw[['split', 'season']].value_counts().sort_index().to_frame('rows')

131,259 rows x 98 columns
6,150 games, 1,029 players, seasons [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


rows
split season       
test  2025    26619
train 2021    26039
      2022    25894
      2023    26401
val   2024    26306

## Step 1: drop rows with 0 minutes

A 0-minute row is a player who was on the game sheet but did not play. Their `target_usg_pct` is undefined, a few still carry a stat (a sub-minute stint rounded down), and some are marked as starters. They are not a useful target or training example for a points model, so they are dropped.

Rows with 1 to 5 minutes are **kept**. They are real appearances, and removing them based on same-game minutes would bias which games the model sees.

Note on the other columns: the pre-game features of later games were computed with these appearances included (a 0-minute game counts in `min_ewm`, `prior_games`, and so on), and that is left as is. Because of this, `games_played_season` and `prior_games` can be larger than the number of rows a player has left in this file.

In [3]:
stat_cols = ['target_pts', 'target_fga', 'target_fta', 'target_reb', 'target_ast',
             'target_stl', 'target_blk', 'target_tov', 'target_pf']

drop_mask = df['target_min'] <= MIN_TARGET_MINUTES
dropped = df[drop_mask]
print(f'Rows dropped (target_min <= {MIN_TARGET_MINUTES}): {len(dropped):,}')
print(f'  with a stray stat > 0:  {(dropped[stat_cols].sum(axis=1) > 0).sum():,}')
print(f"  marked as starters:     {int(dropped['target_starter'].sum()):,}")
print(f"  split breakdown:        {dropped['split'].value_counts().to_dict()}")

df = df.loc[~drop_mask].reset_index(drop=True)
N_DROPPED = len(raw) - len(df)
record('1. drop 0-minute rows', rows_dropped=N_DROPPED, note=f'target_min <= {MIN_TARGET_MINUTES}')
print(f'Rows left: {len(df):,}')

Rows dropped (target_min <= 0): 457
  with a stray stat > 0:  71
  marked as starters:     7
  split breakdown:        {'train': 278, 'val': 100, 'test': 79}
Rows left: 130,802


## Step 2: one `espn_player_id` per player

One `player_id` (Trevon Scott) appears with two different ESPN IDs. Every player is given the ESPN ID that appears most often in their rows. The code finds any such player, so it also works if the data gets rebuilt.

In [4]:
ids_per_player = df.groupby('player_id')['espn_player_id'].nunique()
multi = ids_per_player[ids_per_player > 1].index
print('player_ids with more than one espn_player_id:', list(multi))
print(df[df['player_id'].isin(multi)].groupby(['player_id', 'player_name', 'espn_player_id']).size().to_string())

canonical = df.groupby('player_id')['espn_player_id'].agg(lambda s: s.value_counts().idxmax())
new_espn = df['player_id'].map(canonical)
changed = int((new_espn != df['espn_player_id']).sum())
df['espn_player_id'] = new_espn.astype(df['espn_player_id'].dtype)

# The reverse must also hold: no ESPN id may belong to two different players.
players_per_espn = df.groupby('espn_player_id')['player_id'].nunique()
assert (players_per_espn == 1).all(), 'An espn_player_id maps to more than one player_id'
record('2. unify espn_player_id', values_changed=changed, note=f'{len(multi)} player(s) had >1 ESPN id')
print(f'Values changed: {changed}')

player_ids with more than one espn_player_id: [1630286]
player_id  player_name   espn_player_id
1630286    Trevon Scott  3934620           6
                         4711005           2
Values changed: 2


## Step 3: collapse `position` to G / F / C

`position` mixes broad codes (G, F, C) with detailed ones (PG, SG, SF, PF). The detailed codes are common in 2021 and almost gone by 2025, so as a feature the column would mostly measure the season. `pos_group` already holds the broad group for every row, so `position` is mapped onto the same three values. The column is kept so the schema does not change.

In [5]:
print('position by season, before:')
print(pd.crosstab(df['season'], df['position']).to_string())

POSITION_MAP = {'PG': 'G', 'SG': 'G', 'SF': 'F', 'PF': 'F'}
before = df['position'].copy()
df['position'] = df['position'].replace(POSITION_MAP)
changed = int((before != df['position']).sum())
record('3. collapse position', values_changed=changed, note='PG/SG -> G, SF/PF -> F')

print('\nposition by season, after:')
print(pd.crosstab(df['season'], df['position']).to_string())
assert (df['position'] == df['pos_group']).all(), 'position and pos_group disagree'
print(f'\nValues changed: {changed:,}')

position by season, before:
position     C     F      G   PF    PG    SF    SG
season                                            
2021      3745  7621   9009  890  1347  1641  1702
2022      3609  8843   9649  572  1057  1211   877
2023      3631  9563  10880  272   665   804   468
2024      3858  9457  12026   67   310   203   285
2025      3966  9682  12860    0    28     4     0



position by season, after:


position     C      F      G
season                      
2021      3745  10152  12058
2022      3609  10626  11583
2023      3631  10639  12013
2024      3858   9727  12621
2025      3966   9686  12888

Values changed: 12,403


## Step 4: clip rate features that blow up on tiny samples

A player with one 1-minute stint and 2 shots has an enormous usage % or points-per-minute. Those values then feed the EWM and season-average features of later games. The caps below are set well above realistic values, so they only touch the extreme tail:

| column | allowed range | reason |
|---|---|---|
| `usg_pct_ewm`, `usg_pct_season_avg` | 0 to 60 | the 99th percentile is about 35 |
| `ts_pct_ewm`, `ts_pct_season_avg` | 0 to 1.0 | a season-long true shooting % above 100% is not real |
| `pts_per_min_ewm` | 0 to 1.5 | the 99th percentile is about 0.9 |
| `fouls_per_min_ewm` | 0 to 0.5 | at most one foul every two minutes |
| `target_usg_pct` | 0 to 100 | usage % is a share of possessions; values above 100 only come from 1-2 minute games |

`target_ts_pct` is **not** clipped. Its maximum of 1.5 is a real value (3 points on 1 shot), and it is undefined when a player took no shots.

In [6]:
CLIPS = {
    'usg_pct_ewm': (0, 60),
    'usg_pct_season_avg': (0, 60),
    'ts_pct_ewm': (0, 1.0),
    'ts_pct_season_avg': (0, 1.0),
    'pts_per_min_ewm': (0, 1.5),
    'fouls_per_min_ewm': (0, 0.5),
    'target_usg_pct': (0, 100),
}

rows = []
for column, (low, high) in CLIPS.items():
    before = df[column]
    after = before.clip(low, high)
    changed = int((before.notna() & (before != after)).sum())
    rows.append({'column': column, 'low': low, 'high': high, 'max before': before.max(),
                 'max after': after.max(), 'values clipped': changed})
    df[column] = after
    record(f'4. clip {column}', values_changed=changed, note=f'[{low}, {high}]')

pd.DataFrame(rows).set_index('column')

,low,high,max before,max after,values clipped
column,,,,,
usg_pct_ewm,0,60.0,162.3816,60.0,18
usg_pct_season_avg,0,60.0,162.3816,60.0,22
ts_pct_ewm,0,1.0,1.5000,1.0,134
ts_pct_season_avg,0,1.0,1.5000,1.0,294
pts_per_min_ewm,0,1.5,3.0000,1.5,11
fouls_per_min_ewm,0,0.5,1.0000,0.5,18
target_usg_pct,0,100.0,196.2067,100.0,18


## Step 5: cap `days_since_last_game` at 30

`days_since_last_game` counts calendar days since the player's last game. It is 191 days or more at the start of every season and reached 1,559 for players who returned after years away, which makes it mostly a "first game of the season" marker with a huge range. Capping it at 30 keeps the meaning (long layoff) and removes the extreme range. A season opener is still identifiable through `games_played_season == 0`.

`days_rest` is the team-level rest. It is already capped at 10 by the builder and is not changed.

In [7]:
DAYS_SINCE_CAP = 30

before = df['days_since_last_game']
print('values above the cap:', int((before > DAYS_SINCE_CAP).sum()), '| max before:', before.max())
df['days_since_last_game'] = before.clip(upper=DAYS_SINCE_CAP)
changed = int((before.notna() & (before != df['days_since_last_game'])).sum())
record('5. cap days_since_last_game', values_changed=changed, note=f'upper={DAYS_SINCE_CAP}')
print('max after:', df['days_since_last_game'].max(), '| values changed:', changed)

values above the cap: 2553 | max before: 1559.0
max after: 30.0 | values changed: 2553


## Step 6: winsorize team and opponent ratings

Team and opponent ratings are season-to-date numbers. In a team's first few games they come from 1-3 games, so they swing far outside normal range (for example an offensive rating of 77 against a typical 112). These columns are clipped at their 0.5th and 99.5th percentiles.

The bounds are computed on the **train split only** and then applied to every row, so nothing from validation or test influences the cleaning.

In [8]:
WINSOR_COLUMNS = [
    'team_off_rating', 'team_def_rating', 'team_pace',
    'opp_off_rating', 'opp_def_rating', 'opp_pace',
    'opp_pts_allowed_pos', 'opp_pts_allowed_pos_rel', 'exp_team_pts',
]
LOWER_Q, UPPER_Q = 0.005, 0.995

train_rows = df['split'] == 'train'
rows = []
for column in WINSOR_COLUMNS:
    low, high = df.loc[train_rows, column].quantile([LOWER_Q, UPPER_Q])
    before = df[column]
    after = before.clip(low, high)
    changed = int((before.notna() & (before != after)).sum())
    rows.append({'column': column, 'lower bound': low, 'upper bound': high, 'min before': before.min(),
                 'max before': before.max(), 'values clipped': changed})
    df[column] = after
    record(f'6. winsorize {column}', values_changed=changed, note=f'train {LOWER_Q:.1%}-{UPPER_Q:.1%}')

WINSOR_BOUNDS = pd.DataFrame(rows).set_index('column')
WINSOR_BOUNDS

,lower bound,upper bound,min before,max before,values clipped
column,,,,,
team_off_rating,96.2335,124.3900,77.5266,127.6325,989
team_def_rating,95.6479,122.9829,77.5266,127.3631,1617
team_pace,95.1804,107.5080,93.3180,115.2800,1546
opp_off_rating,96.3930,124.4514,77.5266,127.6325,972
opp_def_rating,95.8346,122.9829,77.5266,127.3631,1661
opp_pace,95.1624,107.5080,93.3180,115.2800,1566
opp_pts_allowed_pos,8.3000,70.9000,0.0000,92.0000,1262
opp_pts_allowed_pos_rel,0.5121,1.5148,0.0000,2.5551,1122
exp_team_pts,93.1295,133.2182,71.4445,149.9064,1239


## Left as is, and why

These show up in the EDA but are **not** changed here:

- **Missing values.** Almost all of them mean "no history yet", which is information for the model, and XGBoost sends NaN down a learned branch, so there is nothing to fix. The causes:
  - `prev_season_pts_avg`, `prev_season_min_avg`: no data before 2021, plus rookies and players new to the data.
  - `rotation_size`, `rotation_out`, `missing_ppg`, `missing_min`, `top3_scorers_out`: the first 5 games of each team's season.
  - season averages: a player's first game of a season; EWM and last-N features: a player's first game ever.
  - `target_ts_pct`: undefined when a player took no shots and no free throws.
- **`experience_years` (about 12% NaN).** The builder documents this: ESPN omits `debutYear` for 2021+ debuts, so a player who was already on a roster in the first season cannot be classified as rookie or veteran. Filling it would need a different data source, so it has to be fixed in `scripts/build_dataset.py`, not here.
- **Missing Chicago rows in 2025.** 13 Chicago games in the 2025 (test) season are missing some players, so team minutes and points do not add up. The rows that are present are correct. Recovering the missing players needs a re-scrape in `scripts/build_dataset.py`.
- **Synthetic `player_id` values.** IDs of the form `9,000,000,000 + espn_player_id` are documented in `player_id_map.csv` and are valid; they belong to players without an NBA ID.
- **Highly correlated features** (for example `min_ewm` and `team_min_share_ewm`). They are harmless for tree models. Prune them only if you switch to a linear model.
- **Games of 48+ minutes.** These are overtime games and are real.

## Validate the cleaned table

If any assertion fails, the notebook stops before anything is saved.

In [9]:
numeric = df.select_dtypes('number')

# Structure: same columns, same order, same dtypes, only the dropped rows removed.
assert list(df.columns) == ORIGINAL_COLUMNS, 'Columns changed'
assert (df.dtypes == ORIGINAL_DTYPES).all(), df.dtypes[df.dtypes != ORIGINAL_DTYPES]
assert len(df) == len(raw) - N_DROPPED
assert not df.duplicated(['player_id', 'game_id']).any(), 'Duplicate (player_id, game_id)'
assert pd.to_datetime(df['game_date']).is_monotonic_increasing, 'Date order was lost'

# Every value is finite or NaN, and the dropped rows are really gone.
assert np.isfinite(numeric.fillna(0)).all().all(), 'Infinite values'
assert (df['target_min'] > MIN_TARGET_MINUTES).all()

# Model-facing contract: same feature list, no same-game columns among the features.
features = full_features.feature_columns(df.columns)
assert features == ORIGINAL_FEATURES, 'Feature list changed'
assert not any(c.startswith('target_') for c in features)

# Splits are untouched: each season is in exactly one split, same as the original.
assert df.groupby('season')['split'].nunique().eq(1).all()
assert df.groupby('season')['split'].first().to_dict() == ORIGINAL_SPLIT_BY_SEASON

# Box-score identities still hold.
assert (df['target_pts'] == 2 * df['target_fgm'] + df['target_fg3m'] + df['target_ftm']).all()
assert (df['target_reb'] == df['target_oreb'] + df['target_dreb']).all()
assert (df['target_fgm'] <= df['target_fga']).all()
assert (df['target_fg3m'] <= df['target_fg3a']).all()
assert (df['target_ftm'] <= df['target_fta']).all()

# Cleaning rules hold.
for column, (low, high) in CLIPS.items():
    assert df[column].min() >= low and df[column].max() <= high, column
assert df['days_since_last_game'].max() <= DAYS_SINCE_CAP
for column, bounds in WINSOR_BOUNDS.iterrows():
    assert df[column].min() >= bounds['lower bound'] - 1e-9, column
    assert df[column].max() <= bounds['upper bound'] + 1e-9, column
assert set(df['position'].unique()) == {'G', 'F', 'C'}
assert (df['position'] == df['pos_group']).all()
assert df.groupby('player_id')['espn_player_id'].nunique().eq(1).all()

# Rows with missing values are not removed or filled: NaN counts only change where a row was dropped.
print('All checks passed.')
starters = df.groupby(['game_id', 'team_id'])['target_starter'].sum()
print(f'Team-games without exactly 5 starters after step 1: {(starters != 5).sum()} of {len(starters):,}')

All checks passed.


Team-games without exactly 5 starters after step 1: 7 of 12,300


## Summary and save

The table lists every cleaning step with the rows it removed and the values it changed. The cleaned table is then written to `data/NBA_player_stats_cleaned.csv`, with the same columns, order and date format as the original, and reloaded once to confirm it round-trips.

In [10]:
summary = pd.DataFrame(log)
display(summary)
print(f'Total rows: {len(raw):,} -> {len(df):,} ({len(raw) - len(df):,} dropped)')

nan_before = raw.isna().sum()
nan_after = df.isna().sum()
nan_change = pd.DataFrame({'NaN before': nan_before, 'NaN after': nan_after})
nan_change = nan_change[nan_change['NaN before'] != nan_change['NaN after']]
print('\nColumns whose NaN count changed (only because rows were dropped):')
display(nan_change)

,step,rows dropped,values changed,note
0,1. drop 0-minute rows,457,0,target_min <= 0
1,2. unify espn_player_id,0,2,1 player(s) had >1 ESPN id
2,3. collapse position,0,12403,"PG/SG -> G, SF/PF -> F"
3,4. clip usg_pct_ewm,0,18,"[0, 60]"
4,4. clip usg_pct_season_avg,0,22,"[0, 60]"
5,4. clip ts_pct_ewm,0,134,"[0, 1.0]"
6,4. clip ts_pct_season_avg,0,294,"[0, 1.0]"
7,4. clip pts_per_min_ewm,0,11,"[0, 1.5]"
8,4. clip fouls_per_min_ewm,0,18,"[0, 0.5]"
9,4. clip target_usg_pct,0,18,"[0, 100]"


Total rows: 131,259 -> 130,802 (457 dropped)



Columns whose NaN count changed (only because rows were dropped):


,NaN before,NaN after
target_usg_pct,457,0
target_ts_pct,6014,5575
min_ewm,1029,1021
min_last1,1029,1021
min_season_avg,2867,2848
min_std_last10,2033,2010
team_min_share_ewm,1029,1021
fouls_per_min_ewm,1038,1029
starter_rate_last10,1029,1021
pts_ewm,1029,1021


In [11]:
if SAVE_OUTPUT:
    df.to_csv(OUTPUT_PATH, index=False)
    check = pd.read_csv(OUTPUT_PATH, dtype={'game_id': str})
    assert check.shape == df.shape and list(check.columns) == ORIGINAL_COLUMNS
    assert (check.dtypes == ORIGINAL_DTYPES).all(), 'dtypes changed on reload'
    assert check['game_id'].equals(df['game_id'])
    print(f'Wrote {len(df):,} rows x {df.shape[1]} columns to {OUTPUT_PATH}')
else:
    print(f'SAVE_OUTPUT is False; would write {len(df):,} rows to {OUTPUT_PATH}')

Wrote 130,802 rows x 98 columns to /Users/swayamshree/Desktop/Capstone Project/CPSC491-02Group8SportsBetting/data/NBA_player_stats_cleaned.csv
